# Pasture Studies — bounded Kaggle training and readiness

`PREFLIGHT`, `SMOKE`, and `READINESS` retain their diagnostic-only contracts. `EXPLORATORY` is a separately configured, fixed-budget A2C run (256 episodes / 16 updates, one T4, five exploratory rollouts). Its rewards and policy are exploratory only, not formal evidence or eligible for checkpoint selection/promotion.

`EXPLORATORY_COMPARISON` is a separate 24-job maximum, nonformal A-D calibration suite (four variants × three fixed seeds × 512 episodes, run once on one T4 and once as independent jobs on two T4s). `EXPERIMENT` and `FULL_VALIDATION` remain hard-disabled. No pilot/held-out/locked outcomes are accessed; no biological-validity or GPU-acceleration claims.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, time, uuid, zipfile

REPO_URL = "https://github.com/santleme/pasture-studies.git"
# Updated to the reviewed immutable source commit before publication.
PASTURE_DEPLOYMENT = None  # KAGGLE_DEPLOYMENT_CONFIG_MARKER
DEPLOY = PASTURE_DEPLOYMENT
if not isinstance(DEPLOY, dict):
    raise RuntimeError("Notebook was not prepared by scripts/kaggle_deploy.py")
MODE = DEPLOY.get("mode", "")
if MODE in {"EXPERIMENT", "FULL_VALIDATION"}:
    raise RuntimeError("Experimental execution is hard-disabled in this launcher")
if MODE not in {"PREFLIGHT", "SMOKE", "READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON"}:
    raise RuntimeError("Unsupported execution mode")
EXPECTED_COMMIT = DEPLOY.get("source_commit")
if not isinstance(EXPECTED_COMMIT, str) or len(EXPECTED_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in EXPECTED_COMMIT):
    raise RuntimeError("Invalid immutable source commit pin")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RUN_ID = f"{MODE.lower()}-{uuid.uuid4().hex}"
ARTIFACT_ROOT = WORK_ROOT / f"pasture-release-{RUN_ID}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=False)

def run(cmd, *, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True, text=True)

print("Mode:", MODE, "run_id:", RUN_ID, "source SHA:", EXPECTED_COMMIT)
print("Python:", sys.version, "platform:", platform.platform())


## Immutable source checkout and detached release verification

The host-side release manifest binds committed source notebook bytes and the exact injected staged notebook bytes. Kaggle receives only the notebook code file; the deployment config is embedded as a literal and the external manifest remains a detached host attestation. The launcher verifies the pinned source commit and runtime profile.


In [ ]:
REPO_DIR = WORK_ROOT / f"pasture-source-{EXPECTED_COMMIT[:12]}"
if not REPO_DIR.exists():
    # Partial clone plus sparse checkout keeps reserved evaluation artifacts
    # out of the Kaggle runtime; only immutable runtime code is materialized.
    run(["git", "clone", "--filter=blob:none", "--no-checkout", "--no-tags", REPO_URL, REPO_DIR])
run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "--filter=blob:none", "origin", EXPECTED_COMMIT])
run(["git", "-C", REPO_DIR, "sparse-checkout", "init", "--no-cone"])
run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone",
     "/farm_sim/", "/validation/train_torch_mvp1.py", "/validation/profile_torch_training.py", "/scripts/kaggle_release.py",
     "/requirements-training.txt", "/notebooks/pasture_studies_kaggle.ipynb"])
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])
resolved = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if resolved != EXPECTED_COMMIT:
    raise RuntimeError("Pinned source commit mismatch")
if (REPO_DIR / "artifacts").exists() or (REPO_DIR / "validation/kaggle-mvp1").exists():
    raise RuntimeError("Sparse source checkout unexpectedly includes reserved evaluation artifacts")
source_notebook = REPO_DIR / "notebooks/pasture_studies_kaggle.ipynb"
source_notebook_sha = hashlib.sha256(source_notebook.read_bytes()).hexdigest()

os.chdir(REPO_DIR)
import numpy as np
import torch
from importlib.metadata import version

runtime = {"python": platform.python_version(), "numpy": np.__version__,
           "pytorch": torch.__version__, "cuda_runtime": torch.version.cuda,
           "cuda_available": bool(torch.cuda.is_available()),
           "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
           "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print("Observed runtime:", json.dumps(runtime, sort_keys=True))
if source_notebook_sha != DEPLOY.get("source_notebook_sha256"):
    raise RuntimeError("Committed source notebook SHA differs from staged deployment binding")
if MODE in {"READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON"}:
    if runtime != DEPLOY.get("runtime_profile"):
        raise RuntimeError("Observed runtime differs from the staged T4x2 profile")
    if MODE == "READINESS":
        from scripts.kaggle_release import authorize_readiness
        authorize_readiness(DEPLOY.get("readiness_config"))
        if DEPLOY.get("readiness_config", {}).get("locked_test_access") is not False:
            raise RuntimeError("Readiness config must deny locked outcome access")
    else:
        from scripts.kaggle_release import authorize_mode
        authorize_mode(MODE, {"release_manifest_sha256": DEPLOY.get("release_manifest_sha256")},
                       config=DEPLOY.get("exploratory_config"))
        if DEPLOY.get("exploratory_config", {}).get("locked_test_access") is not False:
            raise RuntimeError("Exploratory config must deny locked outcome access")

# Record exact versions of the historical pins for context; READINESS never installs or changes them.
requirements = REPO_DIR / "requirements-training.txt"
print("Historical scientific dependency pins (unchanged):\n", requirements.read_text())


## Runtime preflight and CUDA exercise

The exploratory path verifies the pinned CUDA T4x2 environment, then uses only `cuda:0` for the bounded A2C learner. It does not read preregistered training, validation, pilot, held-out, or locked seed splits. Exploratory artifacts are explicitly nonformal.


In [ ]:
if MODE in {"READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON"}:
    if not DEPLOY.get("gpu_required") or not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("READINESS requires exactly two visible CUDA GPUs")
    names = runtime["gpu_names"]
    if any("T4" not in name for name in names):
        raise RuntimeError(f"Expected T4x2 hardware, found {names}")
    from farm_sim.torch_training import ActorCritic
    model = ActorCritic(value_head=True, seed=37331, dtype=torch.float32).to("cuda:0")
    x = torch.zeros((4, 20), dtype=torch.float32, device="cuda:0")
    logits, values = model(x)
    (logits.square().mean() + values.square().mean()).backward()
    if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()):
        raise RuntimeError("CUDA forward/backward gradient check failed")
    print("CUDA forward/backward: PASS", names)
elif MODE == "PREFLIGHT":
    print("PREFLIGHT complete; no training launched")
elif MODE == "SMOKE":
    if DEPLOY.get("gpu_required") and not torch.cuda.is_available():
        raise RuntimeError("GPU-required SMOKE cannot fall back to CPU")
    smoke_root = ARTIFACT_ROOT / "smoke"
    jobs = [("cpu", "reinforce", "cpu"), ("cpu", "a2c", "cpu")]
    if torch.cuda.is_available(): jobs.append(("cuda", "a2c", "cuda"))
    for condition, algorithm, device in jobs:
        output = smoke_root / condition / algorithm
        run([sys.executable, "-m", "validation.train_torch_mvp1", "--smoke-update",
             "--algorithm", algorithm, "--device", device, "--seed", "37331",
             "--output", output, "--metrics", output / "metrics.jsonl"], cwd=REPO_DIR)
        latest = output / "smoke" / algorithm / "LATEST"
        if not latest.is_file(): raise RuntimeError(f"SMOKE checkpoint missing: {latest}")
        generations = list((latest.parent).glob("generation-*/manifest.json"))
        if len(generations) != 1: raise RuntimeError(f"SMOKE expected one fresh generation: {latest.parent}")
    print("SMOKE pipeline diagnostics completed; no experiment/evaluation was run")


## Two-GPU bounded DDP and interrupted-run recovery

This runs a two-update A2C recovery diagnostic, not an experiment. The first invocation commits one update and checkpoint; the second resumes from `LATEST` and commits the second. Checkpoints must contain readiness-only flags; rank records must show both CUDA devices doing nonzero finite-gradient updates with synchronized model digests. No policy evaluation is run.


In [ ]:
if MODE == "READINESS":
    readiness_root = ARTIFACT_ROOT / "readiness-ddp"
    readiness_root.mkdir()
    diagnostic_seed = 37331  # reserved pipeline diagnostic; not a preregistration split seed
    base = [sys.executable, "-m", "torch.distributed.run", "--standalone",
            "--nproc_per_node=2", "-m", "validation.train_torch_mvp1",
            "--readiness-ddp", "--algorithm", "a2c", "--device", "cuda",
            "--seed", str(diagnostic_seed), "--episodes", "32", "--checkpoint-every", "1",
            "--output", str(readiness_root)]
    started = time.time()
    run(base + ["--max-updates", "1"], cwd=REPO_DIR)
    latest = readiness_root / "readiness-only/a2c/LATEST"
    if not latest.is_file(): raise RuntimeError("First readiness checkpoint was not committed")
    first_generation = latest.read_text().strip()
    run(base + ["--resume", "latest", "--max-updates", "1"], cwd=REPO_DIR)
    evidence_path = readiness_root / "readiness-only/a2c/readiness-evidence.json"
    evidence = json.loads(evidence_path.read_text())
    if evidence.get("benchmark_eligible") is not False or evidence.get("evaluation_performed") is not False or evidence.get("locked_split_accessed") is not False:
        raise RuntimeError("Readiness evidence has unsafe eligibility/access flags")
    updates = evidence.get("updates", [])
    if len(updates) != 2 or [row.get("update") for row in updates] != [1, 2]:
        raise RuntimeError("Interrupted-run recovery did not produce exactly two consecutive updates")
    rank_rows = [record for update in updates for record in update.get("episode_ids_by_rank", [])]
    if {row.get("cuda_device_index") for row in rank_rows} != {0, 1}:
        raise RuntimeError("Both T4 GPUs did not report a readiness update")
    if not all(row.get("gradient_finite") and row.get("optimizer_step_changed_model") for row in rank_rows):
        raise RuntimeError("At least one GPU rank did not perform finite-gradient training work")
    if len({update.get("model_digest") for update in updates}) != 2:
        raise RuntimeError("Expected model state to change between optimizer updates")
    checkpoint = readiness_root / "readiness-only/a2c"
    generations = sorted(checkpoint.glob("generation-*/manifest.json"))
    if len(generations) != 2:
        raise RuntimeError("Expected exactly two committed checkpoint generations after recovery")
    from farm_sim.torch_checkpoint import _is_complete_generation
    for path in generations:
        manifest = json.loads(path.read_text())
        if not _is_complete_generation(path.parent, expected_algorithm="a2c"):
            raise RuntimeError(f"Checkpoint generation failed integrity validation: {path.parent}")
        payload = torch.load(path.parent / "training.pt", map_location="cpu", weights_only=True)
        state = payload.get("state", {})
        if (manifest.get("config", {}).get("protocol") != "readiness-ddp-v1"
                or state.get("readiness_only") is not True
                or state.get("benchmark_eligible") is not False):
            raise RuntimeError("Checkpoint is not marked readiness-only")
    result = {"mode": MODE, "run_id": RUN_ID, "source_commit": EXPECTED_COMMIT,
              "release_manifest_sha256": DEPLOY.get("release_manifest_sha256") if MODE == "READINESS" else None,
              "notebook_sha256": source_notebook_sha, "runtime": runtime,
              "diagnostic_seed": diagnostic_seed, "updates": 2, "resumed_from_generation": first_generation,
              "checkpoint_generations": [p.parent.name for p in generations],
              "gpu_rank_indices": sorted({r["cuda_device_index"] for r in rank_rows}),
              "cuda_forward_backward": "PASS", "locked_outcomes_accessed": False,
              "evaluation_performed": False, "benchmark_eligible": False,
              "elapsed_seconds_diagnostic_only": time.time() - started}
    (ARTIFACT_ROOT / "readiness-report.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\n")
    print(json.dumps(result, indent=2, sort_keys=True))

elif MODE == "EXPLORATORY":
    # Fixed, deliberately modest learning run. It does not call formal evaluators
    # or read any registered split; five generated worlds are descriptive only.
    exploratory_root = ARTIFACT_ROOT / "exploratory-only"
    exploratory_root.mkdir()
    cfg = DEPLOY["exploratory_config"]
    if cfg["episodes"] != 256 or cfg["optimizer_updates"] != 16 or cfg["device_count"] != 1:
        raise RuntimeError("Exploratory budget differs from the fixed nonformal config")
    command = [sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-a2c",
               "--algorithm", "a2c", "--device", "cuda", "--seed", str(cfg["seed"]),
               "--episodes", str(cfg["episodes"]), "--eval-seeds", str(cfg["evaluation_episodes"]),
               "--checkpoint-every", "1", "--output", str(exploratory_root)]
    started = time.time()
    run(command, cwd=REPO_DIR)
    learner_dir = exploratory_root / "exploratory-only" / "a2c"
    # The trainer writes its shared update log at args.output/metrics.jsonl,
    # one level above the per-algorithm checkpoint directory.
    metrics_path = exploratory_root / "exploratory-only" / "metrics.jsonl"
    rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    if len(rows) != 16 or [row.get("update") for row in rows] != list(range(1, 17)):
        raise RuntimeError("Exploratory A2C did not complete its fixed 16 updates")
    if any(row.get("benchmark_eligible") is not False or row.get("formal_eligible") is not False
           or row.get("exploratory_only") is not True for row in rows):
        raise RuntimeError("Exploratory metrics are not safely marked nonformal")
    latest = learner_dir / "LATEST"
    if not latest.is_file(): raise RuntimeError("Exploratory A2C checkpoint pointer is missing")
    from farm_sim.torch_checkpoint import _is_complete_generation
    generations = sorted(learner_dir.glob("generation-*/manifest.json"))
    if len(generations) != 16:
        raise RuntimeError("Expected one integrity-checked checkpoint generation per exploratory update")
    for manifest_path in generations:
        if not _is_complete_generation(manifest_path.parent, expected_algorithm="a2c"):
            raise RuntimeError(f"Exploratory checkpoint integrity failed: {manifest_path.parent}")
    evaluation_path = learner_dir / "evaluation.json"
    evaluation = json.loads(evaluation_path.read_text())
    if (evaluation.get("exploratory_only") is not True or evaluation.get("formal_eligible") is not False
            or evaluation.get("benchmark_eligible") is not False or len(evaluation.get("episodes", [])) != 5):
        raise RuntimeError("Exploratory rollout record has invalid provenance or count")
    if any(not row.get("complete") or row.get("ticks") != 72
           or sum(row.get("action_counts", {}).values()) != 72 * 5
           for row in evaluation["episodes"]):
        raise RuntimeError("Each exploratory world must contain complete 72-tick rollouts for five cattle")
    report = {"mode": MODE, "exploratory_only": True, "formal_eligible": False,
              "benchmark_eligible": False, "locked_outcomes_accessed": False,
              "source_commit": EXPECTED_COMMIT, "notebook_sha256": source_notebook_sha,
              "runtime": runtime, "config": cfg, "updates": len(rows),
              "reward_by_update": [row["batch_return"] for row in rows],
              "action_counts_by_update": [row["batch_action_counts"] for row in rows],
              "invalid_actions_by_update": [row["invalid_actions"] for row in rows],
              "checkpoint_generations": [p.parent.name for p in generations],
              "checkpoint_latest": latest.read_text().strip(),
              "policy_export": str(next(learner_dir.glob("torch-a2c-seed-*.npz"))),
              "five_cattle_rollouts": evaluation["episodes"],
              "elapsed_seconds": time.time() - started}
    (ARTIFACT_ROOT / "exploratory-report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print("Exploratory reward trajectory (nonformal):", json.dumps(report["reward_by_update"]))
    print(json.dumps({k: v for k, v in report.items() if k != "five_cattle_rollouts"}, indent=2, sort_keys=True))


In [ ]:
if MODE == "EXPLORATORY_COMPARISON":
    cfg = DEPLOY["exploratory_config"]
    if (cfg != {"task": "exploratory_a2c_variant_comparison", "algorithm": "a2c",
                "variants": ["A", "B", "C", "D"], "episodes_per_variant_seed": 512,
                "seeds": [9483001, 9483002, 9483003], "training_jobs": 12, "calibration_passes": 2,
                "max_training_jobs": 24, "max_devices": 2, "locked_test_access": False,
                "variant_config": {
                    "A": {"reward_variant": "original", "state_reward_scale": 0.1,
                          "normalize_advantage": False, "entropy_coefficient": 0.0},
                    "B": {"reward_variant": "original", "state_reward_scale": 0.1,
                          "normalize_advantage": True, "entropy_coefficient": 0.01},
                    "C": {"reward_variant": "state_aware", "state_reward_scale": 0.1,
                          "normalize_advantage": False, "entropy_coefficient": 0.0},
                    "D": {"reward_variant": "state_aware", "state_reward_scale": 0.1,
                          "normalize_advantage": True, "entropy_coefficient": 0.01}},
                "formal_eligible": False, "benchmark_eligible": False,
                "checkpoint_promotion": False}):
        raise RuntimeError("Exploratory comparison config differs from its fixed allowlist")
    if (cfg["training_jobs"] * cfg["calibration_passes"] > cfg["max_training_jobs"]
            or cfg["max_training_jobs"] != 24):
        raise RuntimeError("Exploratory comparison exceeds the 24-job guard")
    comparison_root = ARTIFACT_ROOT / "exploratory-comparison"
    comparison_root.mkdir()
    # A small network-update calibration reports CPU and per-T4 timings. It is
    # diagnostic only; full train/eval claims come only from the fixed suite.
    profile_path = comparison_root / "device-profile.json"
    run([sys.executable, "-m", "validation.profile_torch_training", "--devices", "cpu,cuda:0,cuda:1",
         "--batches", "1,128", "--warmups", "3", "--iterations", "5", "--repeats", "3",
         "--output", profile_path], cwd=REPO_DIR)
    profile = json.loads(profile_path.read_text())
    if len(profile.get("results", [])) != 18 or not profile.get("cuda_available"):
        raise RuntimeError("CPU/T4 calibration did not report the expected device phases")

    started_single = time.perf_counter()
    single_env = os.environ.copy()
    single_env["CUDA_VISIBLE_DEVICES"] = "0"
    run([sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-batch",
         "--algorithm", "a2c", "--device", "cuda", "--output", comparison_root / "single-gpu"],
        cwd=REPO_DIR, env=single_env)
    single_seconds = time.perf_counter() - started_single

    # Two independent, single-process A2C runners execute disjoint halves of
    # the same fixed 12-job suite, each explicitly pinned to one T4. No DDP.
    child_envs = []
    child_logs = []
    children = []
    started_dual = time.perf_counter()
    for worker_index, gpu_index in enumerate((0, 1)):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
        log_path = comparison_root / f"dual-worker-{worker_index}.log"
        log_file = log_path.open("w")
        process = subprocess.Popen(
            [sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-batch",
             "--algorithm", "a2c", "--device", "cuda", "--batch-shard-index", str(worker_index),
             "--batch-shard-count", "2", "--output", str(comparison_root / "dual-independent")],
            cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        children.append((worker_index, process, log_file, log_path))
    for worker_index, process, log_file, log_path in children:
        code = process.wait()
        log_file.close()
        if code:
            raise RuntimeError(f"Dual-T4 worker {worker_index} failed; see {log_path}")
    dual_seconds = time.perf_counter() - started_dual

    expected_updates = 512 // 16
    checks = []
    for lane in ("single-gpu", "dual-independent"):
        for variant in ("A", "B", "C", "D"):
            for seed in cfg["seeds"]:
                run_root = comparison_root / lane / "exploratory-comparison" / f"variant-{variant}" / f"seed-{seed}"
                metrics_path = run_root / "metrics.jsonl"
                rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
                if len(rows) != expected_updates or [r.get("update") for r in rows] != list(range(1, expected_updates + 1)):
                    raise RuntimeError(f"{lane} {variant}/{seed} did not complete exactly 512 episodes")
                if any("raw_environment_batch_return" not in r or "training_target_mean_return_to_go" not in r for r in rows):
                    raise RuntimeError("Comparison log must separate raw rewards from shaped training targets")
                if any(r.get("exploratory_only") is not True or r.get("formal_eligible") is not False
                       or r.get("benchmark_eligible") is not False for r in rows):
                    raise RuntimeError("Comparison training log is not marked exploratory-only")
                if any(not {"rollout_seconds", "forward_loss_seconds", "backward_gradient_seconds",
                            "optimizer_seconds", "checkpoint_io_seconds", "batch_wall_seconds"}.issubset(r.get("timing", {}))
                       for r in rows):
                    raise RuntimeError("Comparison training log is missing stage timing")
                latest = run_root / "a2c" / "LATEST"
                if not latest.is_file():
                    raise RuntimeError("Comparison checkpoint recovery pointer is missing")
                from farm_sim.torch_checkpoint import _is_complete_generation
                generations = sorted((run_root / "a2c").glob("generation-*/manifest.json"))
                if len(generations) != expected_updates or any(
                        not _is_complete_generation(item.parent, expected_algorithm="a2c") for item in generations):
                    raise RuntimeError("Comparison checkpoint generation integrity/count check failed")
                checks.append({"lane": lane, "variant": variant, "seed": seed,
                               "updates": len(rows), "latest": latest.read_text().strip()})
    report = {"mode": MODE, "protocol": "exploratory-comparison-v1", "config": cfg,
              "exploratory_only": True, "formal_eligible": False, "benchmark_eligible": False,
              "locked_outcomes_accessed": False, "source_commit": EXPECTED_COMMIT,
              "notebook_sha256": source_notebook_sha, "runtime": runtime,
              "device_profile": str(profile_path), "training_job_guard": 24, "jobs_per_pass": cfg["training_jobs"],
              "calibration_passes": cfg["calibration_passes"],
              "completed_jobs": len(checks), "single_gpu_seconds": single_seconds,
              "dual_independent_gpu_seconds": dual_seconds,
              "dual_over_single_elapsed_ratio_diagnostic_only": single_seconds / dual_seconds,
              "job_checks": checks, "claim_boundary": "Descriptive exploratory profiling only; no formal or GPU-speedup claim."}
    (comparison_root / "exploratory-comparison-report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


## Integrity-checked artifact export

Exploratory export includes the nonformal reward curve, action/resource diagnostics, five five-cattle rollouts, NumPy policy export, and all checkpoint generations. Artifacts are checksummed and explicitly ineligible for formal analysis, benchmark comparison, checkpoint promotion, or acceptance claims.


In [ ]:
if MODE in {"READINESS", "SMOKE", "EXPLORATORY", "EXPLORATORY_COMPARISON"}:
    files = []
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file() and path.name not in {"artifact-manifest.json", "pasture-readiness.zip"}:
            files.append({"path": path.relative_to(ARTIFACT_ROOT).as_posix(),
                          "size_bytes": path.stat().st_size,
                          "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    artifact_manifest = {"schema_version": 1, "mode": MODE,
        "run_id": RUN_ID, "source_commit": EXPECTED_COMMIT,
        "release_manifest_sha256": (DEPLOY.get("release_manifest_sha256")
            if MODE in {"READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON"} else None),
        "notebook_sha256": source_notebook_sha, "runtime": runtime,
        "benchmark_eligible": False, "formal_eligible": False,
        "exploratory_only": MODE in {"EXPLORATORY", "EXPLORATORY_COMPARISON"}, "locked_outcomes_accessed": False,
        "files": files}
    manifest_path = ARTIFACT_ROOT / "artifact-manifest.json"
    manifest_path.write_text(json.dumps(artifact_manifest, indent=2, sort_keys=True) + "\n")
    archive_path = ARTIFACT_ROOT / "pasture-artifacts.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(manifest_path, "artifact-manifest.json")
        for entry in files: archive.write(ARTIFACT_ROOT / entry["path"], entry["path"])
    with zipfile.ZipFile(archive_path) as archive:
        bad = archive.testzip()
        if bad: raise RuntimeError(f"ZIP integrity failed at {bad}")
    print("Artifact directory:", ARTIFACT_ROOT)
    print("Artifact manifest entries:", len(files))
    print("ZIP SHA-256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())
